# Lesson 7 — SQL Fundamentals

**The bridge from Lesson 2.** Back then we loaded messy files into pandas, restored the
properties they had lost, and answered questions with `df[mask]`, `sort_values`, `head`,
`groupby`, and `merge`. This lesson answers the *same* questions in a different language —
**SQL**, the language databases speak.

The through-line for the whole lesson:

> **Every SQL query you will write here has a pandas twin you already wrote in Lesson 2.**
> pandas is now the analogy source. When a SQL keyword looks new, we point at the pandas
> line that does the same thing, and the mystery evaporates.

There is one genuinely new idea underneath the syntax, and it is worth stating up front.
In Lesson 3 you *described the steps*: "start a running total, loop over the rows, add each
one." SQL asks you to **declare the properties of the rows you want** — which columns, which
condition, which order — and lets the database figure out the steps. Same destination,
opposite driving instructions.

```
Lesson 2 (pandas)                     Lesson 7 (SQL)
-----------------                     --------------
df[["a", "b"]]                        SELECT a, b
df[df["x"] > 5]                       WHERE x > 5
df.sort_values("x")                   ORDER BY x
df.head(10)                           LIMIT 10
df.groupby("k")["v"].sum()            GROUP BY k  /  SUM(v)
```

**Setup.** Everything here runs offline with the Python standard library:

- `sqlite3` ships with Python — no install, no server, no network. A SQLite database is
  just a single file on disk.
- `pandas` runs every query through `pd.read_sql`, so each result prints as the same tidy
  table you already know.
- Run the notebook top to bottom — later cells reuse the database and helpers built in the
  first few.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

print("sqlite3 (library):", sqlite3.version, "| SQLite engine:", sqlite3.sqlite_version)
print("pandas           :", pd.__version__)

# A SQLite database is one file. We keep runtime artifacts under data/ (gitignored),
# never in the repo tree — the same discipline as Lesson 2's data/ and output/ folders.
Path("data").mkdir(exist_ok=True)
DB_PATH = Path("data") / "lesson7_retail.sqlite"
print("database file     :", DB_PATH)

## Building the database

The database is built here, in the notebook, from the two pinned CSVs in `course_data/` —
the very same `customers` and `transactions` tables Lesson 2 cleaned and Lesson 3 scanned.
This cell is **self-contained**: it does not depend on any earlier lesson having run. Loading
a DataFrame into a SQL table is one line, `DataFrame.to_sql`, and it is the mirror image of
the `pd.read_sql` we use to read results back.

In [ ]:
# Read the pinned extracts (real UCI Online Retail rows) into DataFrames...
transactions = pd.read_csv("course_data/lesson2_transactions_base.csv")
customers = pd.read_csv("course_data/lesson2_customers_base.csv")

# ...and write each DataFrame into the SQLite file as a table.
con = sqlite3.connect(DB_PATH)
transactions.to_sql("transactions", con, index=False, if_exists="replace")
customers.to_sql("customers", con, index=False, if_exists="replace")

tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name", con
)
print("tables in the database:", list(tables["name"]))
print("transactions rows:", len(transactions), "| customers rows:", len(customers))

We will run a lot of queries, so here is a one-line helper. `sql(query)` hands the query
to the database and returns the answer as a DataFrame — so a SQL result *is* a pandas object
the moment it comes back. We also keep the plain `transactions` DataFrame around, because
every SQL cell below sits next to the pandas twin that computes the same thing.

In [ ]:
def sql(query):
    """Run a SQL query against the retail database; return the result as a DataFrame."""
    return pd.read_sql(query, con)

# The two views of the same 60 rows, side by side for the rest of the lesson:
#   - `transactions`  -> a pandas DataFrame  (Lesson 2's world)
#   - the SQL table    -> queried with sql() (this lesson's world)
sql("SELECT transaction_id, customer_id, transaction_date, quantity, unit_price, source_country "
    "FROM transactions LIMIT 5")

🧭 The columns are exactly the ones you already know: `transaction_id`, `customer_id`,
`transaction_date`, `product`, `quantity`, `unit_price`, `source_country`. A **table** in SQL
is a DataFrame that lives in a file: rows and typed columns. The difference is not the data —
it is how we *ask* about it.

## Unit 1 — SELECT, WHERE, ORDER BY, LIMIT  ·  ~50 min

Four keywords answer most day-one questions. Read each one as a **declaration**: you name the
property you want, the database delivers rows that satisfy it. You never write the loop.

| SQL keyword | Declares...                        | Lesson 2 twin        |
| ----------- | ---------------------------------- | -------------------- |
| `SELECT`    | which columns come back            | `df[["a", "b"]]`     |
| `WHERE`     | which rows qualify (a condition)   | a boolean mask       |
| `ORDER BY`  | the order of the rows              | `df.sort_values`     |
| `LIMIT`     | how many rows to keep              | `df.head`            |

### 1.1 SELECT and LIMIT — choose columns, cap the rows

In [ ]:
# pandas twin: pick columns with a list, cap rows with head().
transactions[["transaction_id", "product", "unit_price"]].head(5)

In [ ]:
# SQL: name the columns after SELECT, cap rows with LIMIT.
sql("""
    SELECT transaction_id, product, unit_price
    FROM transactions
    LIMIT 5
""")

🧭 `SELECT col1, col2` is `df[["col1", "col2"]]`; `SELECT *` is the whole frame; `LIMIT 5`
is `head(5)`. Notice you did not say *how* to fetch five rows — no loop, no counter. You
declared "at most five" and the engine obliged.

### 1.2 WHERE — keep the rows that satisfy a condition

In [ ]:
# pandas twin: a boolean mask keeps rows where the condition is True.
transactions[transactions["quantity"] >= 6][
    ["transaction_id", "quantity", "source_country"]
].head()

In [ ]:
# SQL: WHERE is the boolean mask, written as a condition.
sql("""
    SELECT transaction_id, quantity, source_country
    FROM transactions
    WHERE quantity >= 6
""")

🧭 `WHERE quantity >= 6` is exactly `df[df["quantity"] >= 6]`. Combine conditions with
`AND` / `OR` / `NOT` (SQL spells out the words that pandas wrote as `&` / `|` / `~`), and no
parentheses gymnastics are required.

In [ ]:
# Combined conditions read almost like English.
sql("""
    SELECT transaction_id, quantity, unit_price, source_country
    FROM transactions
    WHERE quantity >= 6 AND source_country = 'EIRE'
""")

### 1.3 ORDER BY — declare the ordering property

In [ ]:
# pandas twin: sort by a column, descending, take the top 5.
transactions.sort_values("unit_price", ascending=False).head(5)[
    ["transaction_id", "product", "unit_price"]
]

In [ ]:
# SQL: ORDER BY <column> DESC, then LIMIT to make a top-N.
sql("""
    SELECT transaction_id, product, unit_price
    FROM transactions
    ORDER BY unit_price DESC
    LIMIT 5
""")

🧭 `ORDER BY unit_price DESC` is `sort_values("unit_price", ascending=False)`; ascending is
the default in both. `ORDER BY` then `LIMIT` is the classic **top-N** pattern — the SQL twin of
`sort_values(...).head(n)` from Lesson 3's leaderboard.

### 1.4 DISTINCT — the unique labels in a column

In [ ]:
# pandas twin: the distinct values of a column (Lesson 2's set-membership x-ray).
transactions["source_country"].drop_duplicates().tolist()

In [ ]:
# SQL: DISTINCT collapses repeats, returning each label once.
sql("SELECT DISTINCT source_country FROM transactions ORDER BY source_country")

🧭 `SELECT DISTINCT source_country` is `df["source_country"].drop_duplicates()` (or
`unique()`). This is the *uniqueness* property from Lesson 1 again: "which distinct labels
exist here?" — asked in one word.

### Declaring vs describing — the mental shift

Put the four keywords together and read the query as a single sentence of *properties*:

In [ ]:
sql("""
    SELECT DISTINCT source_country, quantity
    FROM transactions
    WHERE quantity >= 6
    ORDER BY quantity DESC
    LIMIT 8
""")

🧭 You declared four things at once — *these columns, distinct, where quantity is at least
six, ordered by quantity, at most eight rows* — and never wrote a single step. That is the whole
personality difference between SQL and the Lesson 3 loops: **loops describe the procedure; SQL
declares the result and lets the engine choose the procedure.** The pandas one-liners live in
between — vectorized, but still you stitching the steps together.

## Unit 2 — Aggregates and GROUP BY  ·  ~50 min

Aggregates collapse many rows into one number: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`. On their
own they summarize the whole table; paired with `GROUP BY` they summarize each group — the
exact split-apply-combine you learned as `groupby` in Lesson 2.

### 2.1 Whole-table aggregates

In [ ]:
# pandas twin: reduce a column to a single number.
line_total = transactions["quantity"] * transactions["unit_price"]
print("count :", transactions.shape[0])
print("sum   :", round(line_total.sum(), 2))
print("avg   :", round(transactions["unit_price"].mean(), 2))
print("min   :", transactions["unit_price"].min())
print("max   :", transactions["unit_price"].max())

In [ ]:
# SQL: the same five reductions, in one SELECT.
sql("""
    SELECT COUNT(*)                          AS count,
           ROUND(SUM(quantity * unit_price), 2) AS sum,
           ROUND(AVG(unit_price), 2)         AS avg,
           MIN(unit_price)                   AS min,
           MAX(unit_price)                   AS max
    FROM transactions
""")

🧭 `COUNT(*)` is `len(df)`, `SUM(col)` is `df[col].sum()`, and so on. `AS name` renames the
output column — the SQL twin of naming a result in `agg`. `ROUND(x, 2)` is `round(x, 2)`; it just
keeps the money readable.

### 2.2 GROUP BY — one summary row per group

In [ ]:
# pandas twin: split by country, sum the revenue in each group.
tx = transactions.copy()
tx["revenue"] = tx["quantity"] * tx["unit_price"]
tx.groupby("source_country")["revenue"].sum().round(2)

In [ ]:
# SQL: GROUP BY names the split key; the aggregate applies within each group.
sql("""
    SELECT source_country                       AS country,
           ROUND(SUM(quantity * unit_price), 2) AS revenue,
           COUNT(*)                             AS n_lines
    FROM transactions
    GROUP BY source_country
    ORDER BY revenue DESC
""")

🧭 `GROUP BY source_country` with `SUM(...)` **is** `groupby("source_country")["revenue"].sum()`
— split, apply, combine. The rule that trips everyone once: every column in `SELECT` must either
be **grouped by** or wrapped in an **aggregate**. A bare column that is neither has no single value
per group, so the database refuses to guess.

### 2.3 HAVING vs WHERE — the classic interview trap, head-on

Both filter. The difference is *when* they filter, and interviewers love it because getting it
wrong is silent:

- **`WHERE` filters rows** — it runs **before** grouping, on the raw rows.
- **`HAVING` filters groups** — it runs **after** grouping, on the aggregated result.

You cannot put an aggregate like `SUM(...)` in `WHERE`, because at `WHERE` time the groups do not
exist yet. That is precisely why `HAVING` had to be invented.

In [ ]:
# WHERE first: filter raw rows (drop the cheap lines), THEN group.
sql("""
    SELECT source_country                       AS country,
           ROUND(SUM(quantity * unit_price), 2) AS revenue
    FROM transactions
    WHERE unit_price >= 1.0          -- a condition on each RAW row
    GROUP BY source_country
    ORDER BY revenue DESC
""")

In [ ]:
# HAVING: group everything first, THEN keep only groups whose total clears 100.
sql("""
    SELECT source_country                       AS country,
           ROUND(SUM(quantity * unit_price), 2) AS revenue
    FROM transactions
    GROUP BY source_country
    HAVING SUM(quantity * unit_price) > 100      -- a condition on each GROUP
    ORDER BY revenue DESC
""")

🧭 The pandas twin makes the timing obvious — it is two separate steps, and `HAVING` is
simply the *second* mask:

```python
per_country = tx.groupby("source_country")["revenue"].sum()   # group first
per_country[per_country > 100]                                # HAVING = mask AFTER
```

`WHERE` is the mask you apply to `tx` **before** `groupby`; `HAVING` is the mask you apply to the
grouped result **after**. Netherlands (total 23.41) survives `WHERE` but is dropped by
`HAVING > 100`. Interview answer in one line: **WHERE filters rows, HAVING filters groups.**

### 2.4 COUNT(*) vs COUNT(col), and the first NULL

Real tables have holes. A missing value in SQL is `NULL` — the database's word for "no value
here", the twin of pandas' `NaN`/`NaT` from Lesson 2. To see how the aggregates treat holes we
need a table that *has* some, so we add one now — clearly labeled, and kept **separate** from the
pinned 60 real rows so we never contaminate the real data.

In [ ]:
# --- AUTHORED TEACHING FIXTURE (not real UCI data) -------------------------
# Eight hand-made order rows with deliberate gaps (NULLs) in country, quantity,
# and order_date. Kept in its OWN table, orders_raw, so the real `transactions`
# table stays pristine. The customer ids are borrowed from the real customers so
# the story stays coherent; every value here is invented for teaching.
orders_raw = pd.DataFrame({
    "order_id":    [1, 2, 3, 4, 5, 6, 7, 8],
    "customer_id": ["C15311", "C14911", "C14646", "C12748",
                    "C13089", "C15311", "C14606", "C14298"],
    "country":     ["United Kingdom", "EIRE", None, "United Kingdom",
                    None, "United Kingdom", "United Kingdom", "EIRE"],
    "quantity":    [6, None, 4, 2, None, 3, 10, 1],
    "unit_price":  [2.50, 3.00, 1.25, 5.00, None, 1.95, 0.85, 7.95],
    "order_date":  ["2011-01-05", "2011-01-20", "2011-02-10", None,
                    "2011-02-28", "2011-03-02", None, "2011-03-15"],
})
orders_raw.to_sql("orders_raw", con, index=False, if_exists="replace")
sql("SELECT * FROM orders_raw")

In [ ]:
# COUNT(*) counts ROWS. COUNT(column) counts NON-NULL values in that column.
sql("""
    SELECT COUNT(*)          AS all_rows,
           COUNT(country)    AS with_country,
           COUNT(quantity)   AS with_quantity,
           COUNT(order_date) AS with_date
    FROM orders_raw
""")

🧭 Eight rows, but only six have a country, six a quantity, six a date. `COUNT(*)` asks "how
many rows?"; `COUNT(country)` asks "how many rows have a country?" — it **skips NULLs**. This is
not new: in Lesson 2, `df["col"].count()` also skipped `NaN` while `len(df)` did not. Same rule,
same trap. `SUM` and `AVG` skip NULLs too, so `AVG(quantity)` divides by 6 (the non-null count),
not 8 — a silent gotcha when a "missing" quietly means "zero".

In [ ]:
# pandas twin: len() vs .count() — the exact same distinction.
print("len(orders_raw)              :", len(orders_raw))          # COUNT(*)
print("orders_raw['country'].count():", orders_raw["country"].count())  # COUNT(country)
print("avg quantity (skips NULL)    :", round(orders_raw["quantity"].mean(), 4))

---
## ⏱️ Session 2 warm-up  ·  5 questions from last time

Answer from memory first, then check.

1. Name the pandas twin of each: `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`.
2. What does it mean to say SQL "declares" a result instead of "describing" it?
3. `WHERE` vs `HAVING`: which filters rows, which filters groups, and which one can hold a
   `SUM(...)`?
4. A table has 8 rows; 2 have a `NULL` in `country`. What do `COUNT(*)` and `COUNT(country)`
   return?
5. Every column in a `GROUP BY` query's `SELECT` must be one of two things. Which two?

<details><summary><b>Answers</b></summary>

1. `SELECT` → column selection `df[[...]]`; `WHERE` → boolean mask; `ORDER BY` → `sort_values`;
   `LIMIT` → `head`.
2. You state the *properties* of the rows you want (columns, condition, order, count); the engine
   chooses the steps. Loops instead spell out the procedure yourself.
3. `WHERE` filters raw rows (before grouping); `HAVING` filters groups (after aggregating). Only
   `HAVING` can hold an aggregate like `SUM(...)` — at `WHERE` time the groups don't exist yet.
4. `COUNT(*)` → 8 (all rows); `COUNT(country)` → 6 (non-null values only).
5. Either **grouped by**, or wrapped in an **aggregate** (`SUM`, `COUNT`, `AVG`, ...).
</details>

## Unit 3 — NULL and dates  ·  ~40 min

Two topics that quietly decide whether your numbers are right: how `NULL` behaves in comparisons,
and how to slice time out of a date. Both have Lesson 2 twins — `isna`/`fillna`, and
`dt.to_period("M")`.

### 3.1 Three-valued logic — NULL is not a value, it is the absence of one

Ordinary logic has two answers: TRUE and FALSE. Add `NULL` and you get a **third**: UNKNOWN. Any
comparison with `NULL` returns UNKNOWN, because "is an unknown value equal to 'EIRE'?" has no honest
answer. And `WHERE` keeps a row only when its condition is **TRUE** — never when it is UNKNOWN.

In [ ]:
# Two filters that look like exact opposites...
uk    = sql("SELECT COUNT(*) AS n FROM orders_raw WHERE country =  'United Kingdom'")
notuk = sql("SELECT COUNT(*) AS n FROM orders_raw WHERE country <> 'United Kingdom'")
print("country =  'United Kingdom' :", int(uk.loc[0, "n"]))
print("country <> 'United Kingdom' :", int(notuk.loc[0, "n"]))
print("rows in table               :", len(orders_raw))

🧭 4 plus 2 is **6, not 8**. The two rows with `NULL` country satisfy *neither* filter — for
them, both `country = 'United Kingdom'` and `country <> 'United Kingdom'` evaluate to UNKNOWN, and
`WHERE` drops UNKNOWN rows. This is the single most common `NULL` bug: a `<>` filter that silently
loses the rows you never accounted for. In Lesson 2 the same thing happened when a `NaN` region
matched no equality test.

### 3.2 IS NULL and COALESCE — find the holes, then fill them on purpose

In [ ]:
# To test for NULL you must use IS NULL / IS NOT NULL -- never = NULL (that is UNKNOWN).
sql("SELECT order_id, customer_id FROM orders_raw WHERE country IS NULL")

In [ ]:
# COALESCE(x, fallback) returns the first non-NULL argument: the SQL twin of fillna().
sql("""
    SELECT order_id,
           COALESCE(country, 'Unknown') AS country,
           COALESCE(quantity, 0)        AS quantity
    FROM orders_raw
    ORDER BY order_id
""")

🧭 `IS NULL` is `df["col"].isna()`; `COALESCE(country, 'Unknown')` is
`df["country"].fillna("Unknown")`. And the Lesson 2 discipline still applies: filling is a
**decision**, not a reflex. `COALESCE(quantity, 0)` is only honest if a missing quantity truly
means zero — otherwise you have invented data, exactly the trap Lesson 2 warned about with missing
prices.

### 3.3 Dates — slicing time out of a timestamp

Our `transactions` carry real dates (`2010-12-01` ... `2011-08-30`). SQLite stores them as text,
and its date functions read that text. The one we need most is `strftime`, which formats a date —
`strftime('%Y-%m', date)` keeps just the year and month, which is exactly **monthly truncation**:
collapsing every day in a month to one bucket.

In [ ]:
# strftime pulls date parts out of the timestamp text.
sql("""
    SELECT transaction_id,
           transaction_date,
           strftime('%Y',    transaction_date) AS year,
           strftime('%Y-%m', transaction_date) AS month
    FROM transactions
    ORDER BY transaction_date
    LIMIT 5
""")

In [ ]:
# pandas twin: to_period("M") is the same monthly truncation (Lesson 2, Unit 7).
tx["month"] = pd.to_datetime(tx["transaction_date"]).dt.to_period("M").astype(str)
tx[["transaction_id", "transaction_date", "month"]].head(5)

### 3.4 Revenue by month — group on the truncated date

In [ ]:
# SQL: truncate to month, then GROUP BY that bucket -- the revenue-by-month report.
sql("""
    SELECT strftime('%Y-%m', transaction_date) AS month,
           ROUND(SUM(quantity * unit_price), 2) AS revenue,
           COUNT(*)                             AS n_lines
    FROM transactions
    GROUP BY month
    ORDER BY month
""")

In [ ]:
# pandas twin: group on the month column, sum revenue.
tx.groupby("month")["revenue"].agg(revenue="sum", n_lines="count").round(2)

🧭 `GROUP BY strftime('%Y-%m', date)` is `groupby(df["date"].dt.to_period("M"))`. December
2010 dominates because that is where most of these pinned rows fall. Notice what you just built:
group a *derived* value (the month), not a stored column — the same move as grouping a pandas
Series you computed on the fly. **This is the shape of the exercise you are about to do**, with one
more grouping key (country) added.

## Unit 4 — Reading queries  ·  ~35 min

A query is written top-to-bottom but the database does **not** run it in that order. Learning the
real order turns SQL from spell-casting into something you can read and predict.

### The logical evaluation order

```
1. FROM       pick the table (and joins)          -> the raw rows exist
2. WHERE      keep rows matching a row condition   -> filter BEFORE grouping
3. GROUP BY   fold rows into groups                -> groups now exist
4. HAVING     keep groups matching a condition     -> filter AFTER grouping
5. SELECT     compute the output columns & aliases -> aliases are born HERE
6. ORDER BY   sort the finished rows
   LIMIT      keep the first n
```

Two consequences fall straight out of this order, and they explain the two things beginners find
"weird":

- **An aggregate cannot appear in `WHERE`.** `WHERE` (step 2) runs before `GROUP BY` (step 3), so
  `SUM(...)` has nothing to sum yet. That is why `HAVING` (step 4) exists.
- **A `SELECT` alias cannot be trusted in `WHERE`.** The alias is created at step 5; `WHERE` at
  step 2 has never heard of it.

**Demonstration 1 — an aggregate in `WHERE` is a genuine error.** We wrap it in `try/except`
so the notebook keeps running (the same way Lesson 2 caught a bad file path).

In [ ]:
try:
    sql("""
        SELECT source_country, SUM(quantity * unit_price) AS revenue
        FROM transactions
        WHERE SUM(quantity * unit_price) > 100
        GROUP BY source_country
    """)
except Exception as error:
    print(type(error).__name__)
    print("->", error)
    print()
    print("Diagnosis: WHERE runs before GROUP BY, so there is no SUM yet.")
    print("Fix:       move the aggregate condition into HAVING.")

**Demonstration 2 — the alias trap that bites even SQLite.** SQLite is unusually lenient and
*will* let you name a `SELECT` alias in `WHERE` — but standard databases (Postgres, SQL Server,
MySQL) reject it, because of the evaluation order above. Worse, when your alias reuses a real
column's name, `WHERE` silently reads the **original column**, not your computed value. Portable
habit: never reference a `SELECT` alias in `WHERE`.

In [ ]:
# The alias `unit_price` is the DOUBLED price. But WHERE unit_price > 5 filters on the
# ORIGINAL column, then SELECT doubles the survivors -- not what the alias suggests.
trap = sql("""
    SELECT transaction_id, unit_price * 2 AS unit_price
    FROM transactions
    WHERE unit_price > 5
    ORDER BY transaction_id
    LIMIT 4
""")
control = sql("""
    SELECT transaction_id, unit_price
    FROM transactions
    WHERE unit_price > 5
    ORDER BY transaction_id
    LIMIT 4
""")
print("what the query returned (doubled):")
print(trap.to_string(index=False))
print()
print("the rows WHERE actually saw (original price > 5, proving it ignored the alias):")
print(control.to_string(index=False))

🧭 The doubled values `13.5, 17.9, ...` are `2 ×` the *original* prices `6.75, 8.95, ...`, and
every original price is above 5. `WHERE` filtered on the stored column and never touched the alias —
concrete proof that the alias does not exist yet at `WHERE` time. Read queries in evaluation order
and this stops being surprising.

### Predict the output of six queries

For each query below: read it in **evaluation order**, write down what you think it returns, *then*
run the cell. The answer is hidden until you have committed to a prediction. This is the single best
way to internalize how a query executes.

**Query 1.** What will this return?

```sql
SELECT source_country AS country, COUNT(*) AS n
FROM transactions
WHERE quantity >= 6
GROUP BY country
ORDER BY n DESC;
```

<details><summary><b>Prediction</b></summary>

`WHERE quantity >= 6` keeps only the bulk lines *first*, then they are grouped by country and
counted. So `n` is the number of **bulk lines** per country (not all lines): United Kingdom 15,
EIRE 4, Netherlands 2. The point: `WHERE` shrank the rows before `COUNT` ever ran.
</details>

In [ ]:
sql("""
    SELECT source_country AS country, COUNT(*) AS n
    FROM transactions
    WHERE quantity >= 6
    GROUP BY country
    ORDER BY n DESC
""")

**Query 2.** What will this return?

```sql
SELECT source_country AS country, COUNT(*) AS n
FROM transactions
GROUP BY country
HAVING COUNT(*) > 10;
```

<details><summary><b>Prediction</b></summary>

No `WHERE`, so every row is grouped; then `HAVING COUNT(*) > 10` keeps only groups with more than
ten lines. Only **United Kingdom** (48 lines) qualifies — EIRE and Netherlands have 6 each and are
dropped. `HAVING` filtered the *groups*, after counting.
</details>

In [ ]:
sql("""
    SELECT source_country AS country, COUNT(*) AS n
    FROM transactions
    GROUP BY country
    HAVING COUNT(*) > 10
""")

**Query 3.** Will this even run? If so, what comes back?

```sql
SELECT source_country, AVG(quantity) AS avg_q
FROM transactions
WHERE AVG(quantity) > 5
GROUP BY source_country;
```

<details><summary><b>Prediction</b></summary>

It **errors** — "misuse of aggregate function AVG()". `WHERE` (step 2) runs before `GROUP BY`
(step 3), so there is no average to compare against yet. The fix is `HAVING AVG(quantity) > 5`. This
is Demonstration 1 again, now as a reflex.
</details>

In [ ]:
try:
    result = sql("""
        SELECT source_country, AVG(quantity) AS avg_q
        FROM transactions
        WHERE AVG(quantity) > 5
        GROUP BY source_country
    """)
    print(result)
except Exception as error:
    print(type(error).__name__, "->", error)

**Query 4.** What order do the rows come out in, and how many?

```sql
SELECT transaction_id, ROUND(quantity * unit_price, 2) AS line_total
FROM transactions
ORDER BY line_total DESC
LIMIT 3;
```

<details><summary><b>Prediction</b></summary>

The three largest line totals, biggest first: `525.60`, `208.80`, `81.36`. Here the alias
`line_total` **is** legal — `ORDER BY` (step 6) runs *after* `SELECT` (step 5), so by then the alias
exists. Contrast with Query 3, where the alias-referencing filter sat in `WHERE`.
</details>

In [ ]:
sql("""
    SELECT transaction_id, ROUND(quantity * unit_price, 2) AS line_total
    FROM transactions
    ORDER BY line_total DESC
    LIMIT 3
""")

**Query 5.** The alias reuses a column name. What does `WHERE` filter on?

```sql
SELECT transaction_id, quantity * 10 AS quantity
FROM transactions
WHERE quantity >= 12
ORDER BY transaction_id
LIMIT 3;
```

<details><summary><b>Prediction</b></summary>

`WHERE` filters on the **original** `quantity` (the stored column), keeping rows with 12 or more
units — the alias does not exist at `WHERE` time. `SELECT` then multiplies each survivor's quantity
by 10. So the returned `quantity` values are `120` or larger, and each equals `10 ×` a stored value
that was itself `>= 12`. The alias/column name collision is the trap from Demonstration 2.
</details>

In [ ]:
sql("""
    SELECT transaction_id, quantity * 10 AS quantity
    FROM transactions
    WHERE quantity >= 12
    ORDER BY transaction_id
    LIMIT 3
""")

**Query 6.** This one uses the `orders_raw` table with its holes. Which rows come back?

```sql
SELECT order_id, country
FROM orders_raw
WHERE country <> 'United Kingdom';
```

<details><summary><b>Prediction</b></summary>

Only the two **EIRE** rows (`order_id` 2 and 8). The two rows with `NULL` country are **not**
returned: `NULL <> 'United Kingdom'` is UNKNOWN, and `WHERE` keeps only TRUE. Three-valued logic
from Unit 3 — the reason a `<>` filter quietly drops your missing-data rows.
</details>

In [ ]:
sql("""
    SELECT order_id, country
    FROM orders_raw
    WHERE country <> 'United Kingdom'
""")

## Wrap-up

### What you can now do

- Read and write the four workhorse clauses — `SELECT`, `WHERE`, `ORDER BY`, `LIMIT` — and see the
  pandas twin behind each.
- Summarize with `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`, split with `GROUP BY`, and filter groups with
  `HAVING` — knowing exactly how it differs from `WHERE`.
- Handle `NULL` honestly: three-valued logic, `IS NULL`, `COALESCE`, and the `COUNT(*)` vs
  `COUNT(col)` distinction.
- Slice time with `strftime` and build a revenue-by-month report.
- Read a query in **logical evaluation order** and predict its output — including why an aggregate
  in `WHERE` errors and why a `SELECT` alias misbehaves there.

### Five misconceptions this lesson should have broken

| Misconception | Reality |
| --- | --- |
| "SQL runs top to bottom, like a script" | It runs `FROM -> WHERE -> GROUP BY -> HAVING -> SELECT -> ORDER BY` — `SELECT` is nearly last |
| "`WHERE` and `HAVING` are interchangeable" | `WHERE` filters rows before grouping; `HAVING` filters groups after — only `HAVING` holds an aggregate |
| "`COUNT(*)` and `COUNT(col)` are the same" | `COUNT(*)` counts rows; `COUNT(col)` skips `NULL` — just like `len(df)` vs `df[col].count()` |
| "`x <> 'A'` returns everything that isn't 'A'" | Rows where `x` is `NULL` return UNKNOWN and are dropped by both `=` and `<>` |
| "An alias I wrote can be used anywhere" | Aliases are born in `SELECT`; `WHERE` runs earlier and cannot see them (SQLite's leniency is a portability trap) |

### The through-line, one sentence

> *A SQL query declares the properties of the rows you want — which columns, which condition, which
> grouping, which order — and every clause has a pandas twin you already wrote; the only genuinely
> new idea is that the database, not you, chooses the steps.*

### Where each Lesson 2 skill went

- *boolean mask* → `WHERE`
- *`sort_values` / `head`* → `ORDER BY` / `LIMIT`
- *`drop_duplicates`* → `DISTINCT`
- *`groupby(...).agg(...)`* → `GROUP BY` with aggregates
- *`isna` / `fillna`* → `IS NULL` / `COALESCE`
- *`dt.to_period("M")`* → `strftime('%Y-%m', ...)`

**Next:** the exercise in `lesson7_exercise/` — write monthly revenue by country in SQL, write the
same report in pandas, and let `assert_frame_equal` prove the twins agree.

### Extensions

[SQLBolt — interactive SQL lessons](https://sqlbolt.com/) · [pandas "Comparison with SQL" guide](https://pandas.pydata.org/docs/getting_started/comparison/comparison_with_sql.html) · [SQLite date & time functions](https://www.sqlite.org/lang_datefunc.html) · [SQLite SELECT reference](https://www.sqlite.org/lang_select.html)

In [ ]:
# Tidy up: close the database connection (the file in data/ persists, gitignored).
con.close()
print("connection closed. Database file kept at:", DB_PATH)